# Data ingestion

- [x] i have to design or code a function which should be able to handle all the file types
- [ ] for that i have to find out the input and output of drag and drop file input
- [ ] i also have to see how much i can get done through inbuilt langchain docloaders

main docs that i have to focus are csv,json,pdf,html for now.....how can i know the type of input file?...
this can be done through os.path,now what is the type of input i recieve from an input box?

In [1]:
from langchain_community.document_loaders import PyPDFLoader,JSONLoader,TextLoader,BSHTMLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers.parent_document_retriever import ParentDocumentRetriever
from langchain_ollama import OllamaEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
import os
from fastapi import File

In [2]:
load_dotenv()

True

In [3]:
file = open('schemes_structured_documents.json','r')

In [ ]:
def generate_embeddings(file_addr:str):
    try:
        embeddings = OllamaEmbeddings(model="embeddinggemma:300m")
    except:
        print("Error initializing OllamaEmbeddings. Please ensure the Ollama server is running and the model is available.")
        return None
    finally:
        embeddings = GoogleGenerativeAIEmbeddings(model="embeddinggemma:300m")
    return embeddings

def ingest_pdf(file_addr:str):
    loader = PyPDFLoader(file_addr)
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=800,
            chunk_overlap=200,
            separators=[
                "\n## ",      # Major sections
                "\n### ",     # Subsections
                "\n#### ",    # Sub-subsections
                "\n\n",       # Paragraphs
                "\n",         # Lines
                ". ",         # Sentences
                " ",          # Words
            ],
            length_function=len,
        )
    split_docs = text_splitter.split_documents(documents)
    print(f"Number of chunks created: {len(split_docs)}")
    return split_docs

def ingest_json(file_addr:str):
    loader = JSONLoader(
        file_addr, 
        jq_schema='.[] | .brief'
    )
    documents  = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=200,
        separators=[
            "\n## ",      # Major sections
            "\n### ",     # Subsections
            "\n#### ",    # Sub-subsections
            "\n\n",       # Paragraphs
            "\n",         # Lines
            ". ",         # Sentences
            " ",          # Words
        ],
        length_function=len,
    )
    split_docs = text_splitter.split_documents(documents)
    print(split_docs[1])  
    return split_docs

def ingest_txt(file_addr:str):
    loader = TextLoader(file_addr,encoding='utf-8')
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=[
            "\n## ",      # Major sections
            "\n### ",     # Subsections
            "\n#### ",    # Sub-subsections
            "\n\n",       # Paragraphs
            "\n",         # Lines
            ". ",         # Sentences
            " ",          # Words
        ],
        length_function=len,
    )
    split_docs = text_splitter.split_documents(documents)
    documents = []
    for i,chunk in enumerate(split_docs):
        documents.append(chunk)


def ingest_html(file_addr:str):
    loader = BSHTMLLoader(file_addr,open_encoding='utf-8', bs_kwargs={"features": "html.parser"})
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=200,
        separators=[
            "</section>",  # Section breaks
            "</article>",  # Article breaks
            "</div>",      # Div breaks
            "</p>",        # Paragraph breaks
            "\n\n",        # Double newlines
            "\n",          # Single newlines
            ". ",          # Sentences
            " ",           # Words
        ],
        length_function=len,
    )
    split_docs = text_splitter.split_documents(documents)
    print(split_docs[0].page_content.)

In [43]:
def ingest(file_addr:str):
    file_extension = os.path.splitext(file_addr)[1].lower()
    if file_extension == '.pdf':
        ingest_pdf(file_addr)
    elif file_extension == '.json':
        ingest_json(file_addr)
    elif file_extension == '.txt':
        ingest_txt(file_addr)
    elif file_extension in ['.html', '.htm']:
        ingest_html(file_addr)
    else:
        raise ValueError(f"Unsupported file type: {file_extension}")


### now as we have classified the uploaded file, we can handle it accordingly
#### lets start with pdf file

In [44]:
chunks = ingest('central_schemes.txt')
chunks

In [45]:
htmlchunks = ingest('test3.html')
htmlchunks

1 
    

        Private Limited Company

        


Start-ups and growing companies pick this popular business structure because it allows outside funding to be raised easily, limits the liabilities of its shareholders, and enables them to offer employee stock options to attract top talent. As these entities must hold board meetings and file annual returns with the Ministry of Corporate Affairs (MCA), they tend to be viewed with more credibility than an LLP or general partnership.
Features of Private Limited Company
